# Laws-DE LLM Enrichment — Blackwell-Optimized (FlashInfer + FP8 KV + Continuous Batching)

Law-side counterpart of `court_llm_descriptor_extractor_blackwell_optimized__3__(1).ipynb`.
Targets the full 175,933-row `laws_de.csv` corpus via the slim input JSONL produced by
`scripts/build_law_llm_input.py` (`artifacts/law_llm_input.jsonl`, ~189 MB, 173,033 rows after
min-char filtering). Static fields (citation, structural article/units/law_code, title metadata,
enactment year, anchors, adjacency, dictionary labels) are NEVER re-derived by the LLM —
they are already on each input row and merged back in by `scripts/merge_law_llm_into_v1_cards.py`.

**Schema (12 LLM fields, JSON only):** `english_summary`, `legal_rule`, `applicability_conditions`,
`exceptions_or_limitations`, `legal_question`, `concepts_en`, `terms_de_to_en` (verbatim DE → Swiss-legal
English equivalent, NOT literal translation), `defined_terms`, `addressees`,
`sanctions_or_consequences`, `provision_role_llm`, `specificity_score`.

**Throughput target:** Qwen3-8B-AWQ on a single RTX PRO 6000 Blackwell (95 GB) with FlashInfer-or-FA4 +
FP8 KV cache + continuous batching. Estimated ~60–90 min for 173k rows (text is much shorter than
court considerations: median 181 chars vs ~600+).

**Quality contract:**
1. Original German legal terms preserved verbatim in `terms_de_to_en[].de` (substring of source).
2. English column is the **Swiss-legal English equivalent** (e.g. `Bewilligung` → "permit / authorisation"),
   not a literal translation.
3. Rule fields (`legal_rule`, `applicability_conditions`, `exceptions_or_limitations`, `legal_question`)
   are emitted only for substantive provisions; boilerplate (commencement, fee tables, annex lists,
   repeals) gets empty rule fields and a low `specificity_score`.
4. No invented citations, dates, BGE numbers, or party names.

**Run order:**
1. Cell 0: Colab/Kaggle setup, package install, FlashInfer install. RESTART RUNTIME ONCE after first run.
2. Cell 1–2: Imports, GPU probe, config.
3. Cell 3: Load `law_llm_input.jsonl` (slim per-row payload).
4. Cell 4: Schema + system/user prompt templates (sophisticated Swiss-legal vocabulary).
5. Cell 5: JSON parsing + descriptor normalization (mirrors `scripts/run_law_llm_enrichment.py`).
6. Cell 6: vLLM load (single attempt, no fallback re-imports).
7. Cell 7: Generation helpers + warm-up.
8. Cell 8: Run extraction (mega-batched, local-disk hot path, append-mode checkpointing).
9. Cell 9: QC pass.

In [1]:
# Cell 0 — Colab/Kaggle setup, package install, FlashInfer install
#
# IMPORTANT: After this cell runs the FIRST time in a fresh Colab/Kaggle runtime,
# RESTART THE RUNTIME ONCE (Runtime → Restart runtime), then re-run from Cell 1.
# This forces a clean Python process so vLLM's and FlashInfer's CUDA extensions
# bind cleanly. Without this, you may hit "LayerName already registered" or
# "Engine core initialization failed" errors.

import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

IN_KAGGLE = bool(os.environ.get('KAGGLE_URL_BASE') or Path('/kaggle').exists())

print('Running in Colab :', IN_COLAB)
print('Running in Kaggle:', IN_KAGGLE)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# --- 1. Core packages -----------------------------------------------------
print('\nInstalling/upgrading core packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'vllm>=0.10.0',
    'transformers>=4.51.0',
    'accelerate', 'safetensors', 'pandas', 'tqdm', 'huggingface_hub',
], check=True)

# --- 2. FlashInfer (3-package model from official docs) ------------------
def _detect_cuda_index_suffix() -> str | None:
    try:
        import torch
    except Exception:
        return None
    cuda = (torch.version.cuda or '').strip()
    if not cuda:
        return None
    parts = cuda.split('.')
    if len(parts) < 2:
        return None
    try:
        major, minor = int(parts[0]), int(parts[1])
    except ValueError:
        return None
    suffix = f'cu{major}{minor}'
    supported = {'cu126', 'cu128', 'cu129', 'cu130', 'cu131'}
    if suffix in supported:
        return suffix
    candidates = sorted(supported, key=lambda s: int(s[2:]))
    detected_int = major * 10 + minor
    best = None
    for c in candidates:
        if int(c[2:]) <= detected_int:
            best = c
    return best or candidates[0]

cuda_idx = _detect_cuda_index_suffix()
print(f'\nDetected CUDA index suffix for FlashInfer: {cuda_idx}')

def _install_flashinfer() -> bool:
    print('Step A: installing flashinfer-python + flashinfer-cubin (pinned together)...')
    r = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'flashinfer-python', 'flashinfer-cubin',
    ])
    if r.returncode != 0:
        print('  flashinfer -python + -cubin install failed')
        return False
    if cuda_idx is not None:
        print(f'Step B: installing flashinfer-jit-cache for {cuda_idx}...')
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '-q', '-U', '--pre',
            'flashinfer-jit-cache',
            '--index-url', f'https://flashinfer.ai/whl/{cuda_idx}',
        ])
    os.environ['FLASHINFER_DISABLE_VERSION_CHECK'] = '1'
    try:
        import importlib
        importlib.invalidate_caches()
        import flashinfer
        print(f'  flashinfer {getattr(flashinfer, "__version__", "?")} importable')
        return True
    except Exception as exc:
        print(f'  flashinfer import failed: {exc!r}')
        return False

flashinfer_ok = _install_flashinfer()
print(f'\nFlashInfer available: {flashinfer_ok}')
if not flashinfer_ok:
    print('NOTE: vLLM will auto-select an attention backend. All other speedups still apply.')

print('\n' + '=' * 60)
print('Setup complete.')
print('If this was the first install in a fresh runtime,')
print('  RESTART RUNTIME ONCE, then resume from Cell 1.')
print('=' * 60)

Running in Colab : True
Running in Kaggle: True
Mounted at /content/drive

Installing/upgrading core packages...

Detected CUDA index suffix for FlashInfer: cu130
Step A: installing flashinfer-python + flashinfer-cubin (pinned together)...
Step B: installing flashinfer-jit-cache for cu130...


  flashinfer 0.6.10 importable

FlashInfer available: True

Setup complete.
If this was the first install in a fresh runtime,
  RESTART RUNTIME ONCE, then resume from Cell 1.


In [1]:
# Cell 1 — Imports and runtime check (does NOT init CUDA in this process)
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Any, Optional
from collections import Counter
import os
import sys
import re
import gc
import ast
import json
import time
import shutil
import subprocess
import traceback

import pandas as pd
from tqdm.auto import tqdm

os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
os.environ.setdefault('FLASHINFER_DISABLE_VERSION_CHECK', '1')

torch = None  # filled in after vLLM has loaded

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get('KAGGLE_URL_BASE') or Path('/kaggle').exists())

if IN_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/swiss_law')
elif IN_KAGGLE:
    BASE_DIR = Path('/kaggle/working/swiss_law')
else:
    BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'
# Checkpoints persisted to Drive (Colab) so a disconnect doesn't lose progress.
LOCAL_SCRATCH = (BASE_DIR / 'data' / 'checkpoints') if IN_COLAB else (BASE_DIR / 'scratch_outputs')
LOCAL_SCRATCH.mkdir(parents=True, exist_ok=True)

print('Imports OK')
print('Running in Colab :', IN_COLAB)
print('Running in Kaggle:', IN_KAGGLE)
print('BASE_DIR        :', BASE_DIR)
print('CHECKPOINT_DIR  :', LOCAL_SCRATCH)

GPU_COMPUTE_CAPABILITY = None
GPU_NAME = None

def _print_gpu_info_via_nvidia_smi():
    global GPU_COMPUTE_CAPABILITY, GPU_NAME
    try:
        out = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=index,name,compute_cap,memory.total,memory.free,driver_version',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=10, check=True,
        )
        for line in out.stdout.strip().splitlines():
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 6:
                idx, name, cc, mem_total, mem_free, drv = parts[:6]
                print(f'GPU {idx}: {name} (CC {cc}); free={int(mem_free)/1024:.2f} GiB / total={int(mem_total)/1024:.2f} GiB; driver {drv}')
                if GPU_COMPUTE_CAPABILITY is None:
                    GPU_COMPUTE_CAPABILITY = cc
                    GPU_NAME = name
    except FileNotFoundError:
        print('WARNING: nvidia-smi not found; assuming no GPU available.')
    except subprocess.CalledProcessError as exc:
        print(f'nvidia-smi failed: {exc!r}')

_print_gpu_info_via_nvidia_smi()

try:
    import flashinfer  # noqa: F401
    HAS_FLASHINFER = True
    print(f'FlashInfer importable: {getattr(flashinfer, "__version__", "?")}')
except Exception as exc:
    HAS_FLASHINFER = False
    print(f'FlashInfer NOT importable: {exc!r}')

FLASHINFER_USABLE = HAS_FLASHINFER
if HAS_FLASHINFER and GPU_COMPUTE_CAPABILITY:
    cc_major = GPU_COMPUTE_CAPABILITY.split('.')[0]
    if cc_major == '12':
        FLASHINFER_USABLE = False
        print(f'\nFlashInfer installed but SM {GPU_COMPUTE_CAPABILITY} ({GPU_NAME}) is consumer/workstation Blackwell.')
        print('FlashInfer 0.6.x has no SM 12.x cubins; vLLM will auto-select FA4 (equally fast).')
print(f'FlashInfer effective: {FLASHINFER_USABLE}')

Imports OK
Running in Colab : True
Running in Kaggle: True
BASE_DIR        : /content/drive/MyDrive/swiss_law
CHECKPOINT_DIR  : /content/drive/MyDrive/swiss_law/data/checkpoints
GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (CC 12.0); free=94.97 GiB / total=95.59 GiB; driver 580.82.07


FlashInfer importable: 0.6.10

FlashInfer installed but SM 12.0 (NVIDIA RTX PRO 6000 Blackwell Server Edition) is consumer/workstation Blackwell.
FlashInfer 0.6.x has no SM 12.x cubins; vLLM will auto-select FA4 (equally fast).
FlashInfer effective: False


In [2]:
# Cell 2 — Config (Blackwell-optimized; quality preserved)

@dataclass
class Config:
    # ---- Paths ----
    base_dir: str = str(BASE_DIR)
    data_dir: str = str(DATA_DIR)
    output_dir: str = str(OUTPUT_DIR)
    model_download_dir: str = str(MODEL_DIR / 'huggingface')
    local_scratch_dir: str = str(LOCAL_SCRATCH)

    # ---- Input ----
    input_jsonl: str = str(DATA_DIR / 'law_llm_input.jsonl')
    fallback_input_jsonl: str = 'law_llm_input.jsonl'

    # ---- Model ----
    model_name: str = 'Qwen/Qwen3-8B-AWQ'

    # ---- Slice ----
    start: int = 0
    limit: int = 0
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 20
    # Law p99 = 1274 chars. 1500 covers >99% untruncated, identical reasoning to before.
    max_text_chars: int = 1500

    # ---- GPU / vLLM ----
    gpu_mode: str = 'single'
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.92

    # System+schema (~1390 tok with multilingual DE/FR/IT vocab + addressees
    # vocabulary, prefix-cached) + user header (~150 tok) + text up to ~750 tok
    # + output 450 tok = ~2740 worst case. 3072 fits with ~330-token margin.
    # If OOM at vLLM load, drop max_num_seqs to 288 (KV per-slot is +33% vs 2304).
    max_model_len: int = 3072

    # Match the court run's concurrency. FP8 KV is identical, so the slot count is too.
    max_num_seqs: int = 384
    submit_chunk: int = 2048

    enforce_eager: bool = False
    quantization: str = 'awq_marlin'
    disable_custom_all_reduce: bool = True

    # FP8 KV — same as court.
    kv_cache_dtype: Optional[str] = None

    # OFF on SM 12.x (same incompatibility found in the court run).
    enable_ngram_speculation: bool = False
    speculative_num_tokens: int = 5
    speculative_ngram_min: int = 2
    speculative_ngram_max: int = 4

    # ---- Sampling / generation ----
    use_structured_outputs: bool = False
    # 12-field schema with up to 10 term pairs + 4 defined terms + 5+5 lists is ~350-400 tok p99.
    # 450 leaves margin for the longest legitimate JSON; do NOT need 700.
    max_new_tokens: int = 450
    retry_max_new_tokens: int = 600
    max_retries: int = 1
    # Slight sampling avoids the literal-translation attractor at T=0
    # (Bewilligung -> "approval" instead of "permit / authorisation"). Keep.
    temperature: float = 0.1
    top_p: float = 0.9
    # Reverted to 1.0 — at T=0.1 the penalty adds decode cost without quality benefit.
    # (Same finding as the court run.)
    repetition_penalty: float = 1.0
    enable_thinking: bool = False

    # ---- Output / debug ----
    include_raw_output_on_success: bool = False

cfg = Config()


In [3]:
# Cell 3 — Load law_llm_input.jsonl
#
# Each input row is the slim payload built by scripts/build_law_llm_input.py.
# We use _source_row as the checkpoint key (stable across runs).

def resolve_input_path() -> Path:
    candidates = [
        Path(cfg.input_jsonl),
        DATA_DIR / cfg.fallback_input_jsonl,
        BASE_DIR / cfg.fallback_input_jsonl,
        Path('/content') / cfg.fallback_input_jsonl,
        Path('/kaggle/input') / cfg.fallback_input_jsonl,
        Path.cwd() / cfg.fallback_input_jsonl,
    ]
    for p in candidates:
        if p.exists():
            return p
    search_roots = [DATA_DIR, BASE_DIR, Path('/content'), Path('/kaggle/input')]
    for root in search_roots:
        if root.exists():
            for pat in ['**/law_llm_input.jsonl']:
                found = sorted(root.glob(pat))
                if found:
                    return found[0]
    raise FileNotFoundError(
        'Could not find law_llm_input.jsonl. Upload it to '
        f'{DATA_DIR / cfg.fallback_input_jsonl} (Drive) or attach the dataset that contains it (Kaggle).'
    )

input_path = resolve_input_path()
print('Using input:', input_path)
print(f'  size: {input_path.stat().st_size / 1024**2:.1f} MB')

rows = []
skipped_no_text = 0
skipped_short = 0
with input_path.open(encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            continue
        text = (rec.get('text') or '').strip()
        citation = (rec.get('citation') or '').strip()
        if not citation or not text:
            skipped_no_text += 1
            continue
        if len(text) < cfg.min_text_chars:
            skipped_short += 1
            continue
        rows.append({
            '_source_row': int(rec.get('_source_row', line_num - 1)),
            'citation': citation,
            'law_title': rec.get('law_title', '') or '',
            'title_section_path': rec.get('title_section_path', '') or '',
            'structural': rec.get('structural', {}) or {},
            'title_metadata': rec.get('title_metadata', {}) or {},
            'static_hints': rec.get('static_hints', {}) or {},
            'llm_priority': rec.get('llm_priority', 'medium'),
            'text': text,
            '_text_len': len(text),
        })

print(f'Loaded: {len(rows):,} valid rows  (skipped: {skipped_no_text:,} missing, {skipped_short:,} below min_chars)')

valid = pd.DataFrame(rows)

desc = valid['_text_len'].describe(percentiles=[0.5, 0.9, 0.95, 0.99])
print('\nText length distribution (chars):')
print(desc.to_string())
over = (valid['_text_len'] > cfg.max_text_chars).sum()
print(f"\nRows exceeding max_text_chars={cfg.max_text_chars}: {over:,} / {len(valid):,} ({100*over/max(len(valid),1):.2f}%)")
print('(those rows get head+tail truncation; quality preserved.)')

print('\nllm_priority mix:')
print(valid['llm_priority'].value_counts().to_string())

if cfg.sample_random:
    pool = valid.iloc[cfg.start:] if cfg.start else valid
    work_df = pool.sample(n=min(cfg.limit, len(pool)) if cfg.limit else len(pool),
                          random_state=cfg.random_seed)
else:
    end = None if not cfg.limit else cfg.start + cfg.limit
    work_df = valid.iloc[cfg.start:end].reset_index(drop=True)

print(f'\nWork slice: {len(work_df):,} rows (start={cfg.start}, limit={cfg.limit or "all"})')
citation_col, text_col = 'citation', 'text'

Using input: /content/drive/MyDrive/swiss_law/data/law_llm_input.jsonl
  size: 188.6 MB
Loaded: 173,033 valid rows  (skipped: 0 missing, 0 below min_chars)

Text length distribution (chars):
count    173033.000000
mean        242.750614
std         237.947206
min          20.000000
50%         184.000000
90%         432.000000
95%         599.000000
99%        1277.680000
max        2500.000000

Rows exceeding max_text_chars=1500: 1,205 / 173,033 (0.70%)
(those rows get head+tail truncation; quality preserved.)

llm_priority mix:
llm_priority
medium    147941
high       21490
low         3602

Work slice: 173,033 rows (start=0, limit=all)


In [5]:
DESCRIPTOR_KEYS = [
    'english_summary',
    'legal_rule',
    'applicability_conditions',
    'exceptions_or_limitations',
    'legal_question',
    'concepts_en',
    'terms_de_to_en',
    'defined_terms',
    'addressees',
    'sanctions_or_consequences',
    'provision_role_llm',
    'specificity_score',
]

PROVISION_ROLES = {
    'definition', 'purpose', 'scope', 'principle',
    'right_or_entitlement', 'duty', 'prohibition', 'procedure',
    'competence', 'sanction_or_penalty',
    'data_reporting', 'fees_or_costs', 'transitional_or_commencement',
    'other',
}
BOILERPLATE_ROLES = {'transitional_or_commencement', 'fees_or_costs', 'data_reporting'}

LLM_SCHEMA_HINT = {
    'english_summary': '<=2 sentences English; what THIS article says (not the law title)',
    'legal_rule': '<=25 words operative rule; empty for boilerplate',
    'applicability_conditions': ['0-5 short English conditions'],
    'exceptions_or_limitations': ['0-5 English carve-outs'],
    'legal_question': '<=18 words; empty for boilerplate',
    'concepts_en': ['3-8 English legal concepts'],
    'terms_de_to_en': [
        {'de': 'first verbatim DE/FR/IT term', 'en': 'Swiss-legal English equivalent'},
        {'de': 'second verbatim term',         'en': 'Swiss-legal English equivalent'},
    ],
    'defined_terms': [
        {'term': 'first verbatim term defined here', 'definition': 'English gloss'},
        {'term': 'second verbatim term',             'definition': 'English gloss'},
    ],
    'addressees': ['0-6 English labels: who is bound'],
    'sanctions_or_consequences': ['0-4 English items in text'],
    'provision_role_llm': 'definition|purpose|scope|principle|right_or_entitlement|duty|prohibition|procedure|competence|sanction_or_penalty|data_reporting|fees_or_costs|transitional_or_commencement|other',
    'specificity_score': '0..1',
}

_SCHEMA_JSON = json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False, separators=(',', ':'))

SYSTEM_PROMPT = f'''You are a Swiss legal-interpretation assistant working on individual articles
of Swiss federal law (Bundesgesetze / lois fédérales / leggi federali, Verordnungen /
ordonnances / ordinanze, the Bundesverfassung / Constitution fédérale / Costituzione
federale, Verträge / traités / trattati, SR-numbered statutes). The article text is
in German, French, or Italian. Your job is to extract the operative legal meaning
into English, preserving the exact original-language legal terms.

Return exactly one compact JSON object with exactly this shape (all keys present):
{_SCHEMA_JSON}

Hard rules:
1. JSON only. No prose, no markdown, no preamble.
2. Use English for ALL semantic fields except `terms_de_to_en[].de` and
   `defined_terms[].term`, which MUST be exact substrings of the source text in
   its ORIGINAL language (DE, FR, or IT). The JSON key stays `de` regardless of
   the source language — it just means "original-language term".
3. Map each original-language term to its **Swiss-legal English equivalent**,
   NOT a literal translation. Use the row matching the source language; never
   mix languages within one term entry.

   German (DE) -> English:
     Bewilligung -> permit / authorisation
     Verfügung -> formal administrative order
     Rechtsbegehren -> prayer for relief
     Zuständigkeit -> jurisdiction / competence
     Aufsichtsbehörde -> supervisory authority
     Inverkehrbringen -> placing on the market
     Tatbestand -> set of facts / elements of the offence
     Beschwerde -> appeal
     Eidgenössisch -> federal (Swiss)
     Bundesrat -> Federal Council

   French (FR) -> English:
     autorisation -> permit / authorisation
     décision -> formal administrative order
     conclusions -> prayer for relief
     compétence -> jurisdiction / competence
     autorité de surveillance -> supervisory authority
     mise sur le marché -> placing on the market
     état de fait -> set of facts / elements of the offence
     recours -> appeal
     fédéral -> federal (Swiss)
     Conseil fédéral -> Federal Council

   Italian (IT) -> English:
     autorizzazione -> permit / authorisation
     decisione -> formal administrative order
     conclusioni -> prayer for relief
     competenza -> jurisdiction / competence
     autorità di vigilanza -> supervisory authority
     immissione in commercio -> placing on the market
     fattispecie -> set of facts / elements of the offence
     ricorso -> appeal
     federale -> federal (Swiss)
     Consiglio federale -> Federal Council

   Acronyms (language-invariant — keep verbatim): EDI / DFI / SBFI / SEFRI /
   FINMA / ESTV / AFC / EJPD / DFJP / DEFR / IFSN / FF / RS / SR.

   When in doubt, prefer EU/UK statutory English over US wording.
3a. EACH term MUST be its OWN object inside the array. Never write
    {{"de":"A","de":"B"}} or {{"de":"A":"B"}}. Correct shape is
    [{{"de":"A","en":"...A"}},{{"de":"B","en":"...B"}}].
    Same rule for defined_terms with `term`/`definition`.
4. Do NOT invent statute citations, BGE numbers, dates, party names, or
   sanctions that are not literally in the text.
5. Do NOT translate literally. If the source is metaphorical or formal, pick
   the recognised legal English term.
6. Never copy the law title into `english_summary`. Describe what THIS article
   says, not what the parent statute is about.
7. For repeal markers ("Aufgehoben" / "Abrogé" / "Abrogato"), commencement
   clauses ("Tritt am ... in Kraft" / "Entre en vigueur le ..." / "Entra in
   vigore il ..."), pure fee tables, annex code lists, or transitional
   provisions:
     - keep `legal_rule`, `applicability_conditions`, `exceptions_or_limitations`,
       `legal_question` empty.
     - still fill `english_summary`, `concepts_en`, `terms_de_to_en`,
       `provision_role_llm`, `specificity_score` (low).
8. Never put `Art.`, `Abs.`, `Buchstabe`, `Ziffer`, `al.`, `let.`, `ch.`,
   `cpv.`, `lett.`, `n.`, or SR numbers into `terms_de_to_en` — those are
   anchors, not legal terms.
9. `addressees` MUST use labels from this controlled vocabulary (pick the
   closest match; pick multiple when the article binds several actors). Only
   add a free-form label when NONE of these fits:
     - Federal Council
     - federal department or office
     - supervisory authority
     - competent cantonal authority
     - competent communal authority
     - court
     - public prosecutor
     - natural person
     - legal entity / undertaking
     - employer
     - employee
     - taxpayer
     - data subject
     - data controller or processor
     - market participant / operator
     - service provider
     - consumer / customer
     - foreign authority or international organisation
10. Cap: 10 terms_de_to_en, 4 defined_terms, 6 addressees, 5 conditions, 5
    exceptions, 8 concepts_en. Trim to the most salient.
11. If a field has nothing to populate, return an empty string or empty list,
    but the key MUST be present. JSON only.'''

USER_TEMPLATE = '''Citation: {citation}
Law title: {law_title}
Section path: {section_path}
Law code: {law_code}   Article: {article}   Units: {units}
Source type: {source_type}   Enactment year: {year}
Static legal area hint: {legal_area_hint}
Static domain hints: {domain_hints}

Article text (verbatim, original language):
"""
{text}
"""

Return JSON only.'''

def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()

def _units_to_str(units: Any) -> str:
    """Coerce structural.units (list of strings, dicts, or mixed) into a flat label string."""
    if not units:
        return ''
    if isinstance(units, str):
        return units
    if not isinstance(units, (list, tuple)):
        return str(units)
    parts = []
    for u in units:
        if u is None:
            continue
        if isinstance(u, str):
            s = u.strip()
        elif isinstance(u, dict):
            # Common shapes:
            #   {"category":"paragraph","marker":"Abs.","value":"1"}  (real corpus)
            #   {"unit":"Abs.","number":"1"} / {"type":"Abs.","value":"1"} / {"label":"Abs. 1"}
            label = (u.get('label') or u.get('text') or '').strip()
            if not label:
                head = (u.get('marker') or u.get('unit') or u.get('type') or u.get('kind') or '').strip()
                tail = u.get('value')
                if tail is None:
                    tail = u.get('number')
                if tail is None:
                    tail = u.get('n')
                tail = '' if tail is None else str(tail).strip()
                label = (head + ' ' + tail).strip() if (head or tail) else json.dumps(u, ensure_ascii=False)
            s = label
        else:
            s = str(u).strip()
        if s:
            parts.append(s)
    return ', '.join(parts)

def _coerce_str(x: Any, max_chars: int = 0) -> str:
    if x is None:
        return ''
    if isinstance(x, str):
        s = x
    elif isinstance(x, (list, tuple)):
        s = ', '.join(_coerce_str(i) for i in x if i)
    elif isinstance(x, dict):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    s = s.strip()
    return s[:max_chars] if max_chars else s

def build_user_prompt(row: dict) -> str:
    structural = row.get('structural') or {}
    title_meta = row.get('title_metadata') or {}
    hints = row.get('static_hints') or {}
    enactment_date = _coerce_str(title_meta.get('enactment_date'))
    year = _coerce_str(title_meta.get('enactment_year')) or (enactment_date[:4] if enactment_date else '')
    return USER_TEMPLATE.format(
        citation=_coerce_str(row.get('citation'), 200),
        law_title=_coerce_str(row.get('law_title'), 300),
        section_path=_coerce_str(row.get('title_section_path'), 200),
        law_code=_coerce_str(structural.get('law_code'), 60),
        article=_coerce_str(structural.get('article'), 40),
        units=_units_to_str(structural.get('units')),
        source_type=_coerce_str(title_meta.get('source_type'), 60),
        year=year,
        legal_area_hint=_coerce_str(hints.get('legal_area_static'), 100),
        domain_hints=', '.join(_coerce_str(d, 80) for d in (hints.get('domain_labels_en') or [])[:5] if d),
        text=trim_text(row.get('text', ''), cfg.max_text_chars),
    )

In [6]:
# Cell 5 — JSON parsing and descriptor normalization
#
# Mirrors scripts/run_law_llm_enrichment.py exactly. Same input → same output.
def _repair_terms_arrays(s: str) -> str:
    """Repair common Qwen3 malformations in terms_de_to_en / defined_terms arrays.

    Two patterns observed in the wild on this run:
      A) {"de":"X":"Y"}         → {"de":"X","en":"Y"}      (extra inline value, en key missing)
      B) {"de":"X","de":"Y"}    → {"de":"X"},{"de":"Y"}    (multiple entries crammed into one obj)
    Both also occur for `term`/`definition` in defined_terms.
    """
    for outer_key, k1, k2 in (
        ('terms_de_to_en', 'de', 'en'),
        ('defined_terms', 'term', 'definition'),
    ):
        m = re.search(rf'"{outer_key}"\s*:\s*\[', s)
        if not m:
            continue
        body_start = m.end()
        depth = 1; in_str = False; esc = False; i = body_start
        while i < len(s) and depth:
            ch = s[i]
            if in_str:
                if esc: esc = False
                elif ch == '\\': esc = True
                elif ch == '"': in_str = False
            else:
                if ch == '"': in_str = True
                elif ch == '[': depth += 1
                elif ch == ']': depth -= 1
            i += 1
        if depth:
            continue
        body = s[body_start:i-1]
        # A) "de":"X":"Y"  →  "de":"X","en":"Y"
        body = re.sub(
            rf'"{k1}"\s*:\s*("(?:[^"\\]|\\.)*")\s*:\s*("(?:[^"\\]|\\.)*")',
            rf'"{k1}":\1,"{k2}":\2',
            body,
        )
        # B) repeated `,"de":` inside one object  →  `},{"de":`
        body = re.sub(rf',\s*"{k1}"\s*:', rf'}},{{"{k1}":', body)
        s = s[:body_start] + body + s[i-1:]
    return s

def extract_json_object(raw: str) -> dict[str, Any]:
    if raw is None:
        raise ValueError('empty model output')
    s = str(raw).strip()
    s = re.sub(r'^\s*```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```\s*$', '', s)
    s = re.sub(r'<think>.*?</think>', '', s, flags=re.I | re.S).strip()

    start = s.find('{')
    if start < 0:
        raise ValueError(f'no JSON object start found: {s[:300]}')

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = s[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        candidate = re.sub(r',\s*([}\]])', r'\1', candidate)
                        try:
                            return json.loads(candidate)
                        except Exception:
                            return ast.literal_eval(candidate)
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = s[start:i+1]
                    for variant in (
                        candidate,
                        re.sub(r',\s*([}\]])', r'\1', candidate),  # trailing-comma fix
                        _repair_terms_arrays(candidate),            # NEW: dup-key / inline-value fix
                        _repair_terms_arrays(re.sub(r',\s*([}\]])', r'\1', candidate)),  # NEW: both
                    ):
                        try:
                            return json.loads(variant)
                        except Exception:
                            pass
                    try:
                        return ast.literal_eval(_repair_terms_arrays(candidate))
                    except Exception:
                        pass
                    raise ValueError(f'no parseable JSON object found: {candidate[:700]}')

    raise ValueError(f'no balanced JSON object found: {s[:700]}')

def clean_str(x: Any, max_chars: int = 240) -> str:
    s = re.sub(r'\s+', ' ', str(x or '')).strip()
    return s[:max_chars].rstrip()

def clean_list(x: Any, max_items: int, max_chars: int = 80) -> list[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, (list, tuple, set)):
        return []
    out, seen = [], set()
    for item in x:
        s = clean_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out

def clean_pair_list(x: Any, max_items: int, key_a: str, key_b: str,
                    max_chars_a: int = 120, max_chars_b: int = 200) -> list[dict]:
    if not isinstance(x, (list, tuple)):
        return []
    out, seen = [], set()
    for item in x:
        if not isinstance(item, dict):
            continue
        a = clean_str(item.get(key_a), max_chars_a)
        b = clean_str(item.get(key_b), max_chars_b)
        if not a or not b:
            continue
        key = a.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append({key_a: a, key_b: b})
        if len(out) >= max_items:
            break
    return out

def normalize_descriptor(obj: dict[str, Any]) -> dict[str, Any]:
    d = {}
    d['english_summary'] = clean_str(obj.get('english_summary'), 400)
    d['legal_rule'] = clean_str(obj.get('legal_rule'), 260)
    d['applicability_conditions'] = clean_list(obj.get('applicability_conditions'), 5, 160)
    d['exceptions_or_limitations'] = clean_list(obj.get('exceptions_or_limitations'), 5, 160)
    d['legal_question'] = clean_str(obj.get('legal_question'), 220)
    d['concepts_en'] = clean_list(obj.get('concepts_en'), 8, 80)
    d['terms_de_to_en'] = clean_pair_list(obj.get('terms_de_to_en'), 10, 'de', 'en')
    d['defined_terms'] = clean_pair_list(obj.get('defined_terms'), 4, 'term', 'definition',
                                         max_chars_a=120, max_chars_b=300)
    d['addressees'] = clean_list(obj.get('addressees'), 6, 80)
    d['sanctions_or_consequences'] = clean_list(obj.get('sanctions_or_consequences'), 4, 160)

    role = clean_str(obj.get('provision_role_llm'), 60).lower().replace(' ', '_').replace('-', '_')
    d['provision_role_llm'] = role if role in PROVISION_ROLES else 'other'
    try:
        d['specificity_score'] = max(0.0, min(1.0, float(obj.get('specificity_score', 0))))
    except Exception:
        d['specificity_score'] = 0.0

    forbidden = {
        'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
        'outcome_signal', 'enrichment_quality', 'anchor_quality_flags',
        'incoming_references', 'outgoing_references', 'adjacent_citations',
        'law_code', 'law_title', 'enactment_date', 'enactment_year',
    }
    for k in forbidden:
        d.pop(k, None)

    # Boilerplate suppression of rule fields (the merger does this too, but doing it
    # here keeps the descriptor file clean).
    if d['provision_role_llm'] in BOILERPLATE_ROLES:
        d['legal_rule'] = ''
        d['applicability_conditions'] = []
        d['exceptions_or_limitations'] = []
        d['legal_question'] = ''
    return d

def empty_descriptor(error: str = '') -> dict[str, Any]:
    return {
        'english_summary': '',
        'legal_rule': '',
        'applicability_conditions': [],
        'exceptions_or_limitations': [],
        'legal_question': '',
        'concepts_en': [],
        'terms_de_to_en': [],
        'defined_terms': [],
        'addressees': [],
        'sanctions_or_consequences': [],
        'provision_role_llm': 'other',
        'specificity_score': 0.0,
        '_descriptor_error': error[:500],
    }

def grounded_terms_pct(terms: list[dict], source_text: str) -> float:
    if not terms:
        return 1.0
    if not source_text:
        return 0.0
    hits = sum(1 for t in terms if t.get('de') and t['de'] in source_text)
    return round(hits / len(terms), 3)

In [7]:
# Cell 6 — Load vLLM (single attempt, no fallback re-imports)
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import inspect

print('Loading tokenizer:', cfg.model_name)

tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name,
    trust_remote_code=True,
    cache_dir=cfg.model_download_dir,
)

_probe_messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'probe'},
]
try:
    _probe_text = tokenizer.apply_chat_template(
        _probe_messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=cfg.enable_thinking,
    )
except TypeError:
    _probe_text = tokenizer.apply_chat_template(
        _probe_messages, tokenize=False, add_generation_prompt=True,
    )
_probe_tokens = tokenizer(_probe_text, return_tensors=None, add_special_tokens=False)['input_ids']
print(f'System+schema prefix size: ~{len(_probe_tokens)} tokens (prefix-cached portion).')

sig_params = set(inspect.signature(LLM).parameters.keys())
print(f'vLLM LLM signature ({len(sig_params)} explicit params); attention_config={"attention_config" in sig_params}')

desired_backend = 'FLASHINFER' if FLASHINFER_USABLE else None

llm_kwargs = dict(
    model=cfg.model_name,
    trust_remote_code=True,
    tensor_parallel_size=cfg.tensor_parallel_size,
    gpu_memory_utilization=cfg.gpu_memory_utilization,
    max_model_len=cfg.max_model_len,
    max_num_seqs=cfg.max_num_seqs,
    enforce_eager=cfg.enforce_eager,
    disable_custom_all_reduce=cfg.disable_custom_all_reduce,
    disable_log_stats=True,
    download_dir=cfg.model_download_dir,
    quantization=cfg.quantization,
)
if cfg.kv_cache_dtype is not None:
    llm_kwargs['kv_cache_dtype'] = cfg.kv_cache_dtype
llm_kwargs['enable_prefix_caching'] = True

backend_method = 'auto-select (vLLM picks FA4 on Blackwell SM 12.x)'
if desired_backend is not None:
    if 'attention_backend' in sig_params:
        llm_kwargs['attention_backend'] = desired_backend
        backend_method = f'attention_backend kwarg = {desired_backend}'
    elif 'attention_config' in sig_params:
        try:
            from vllm.config import AttentionConfig
            llm_kwargs['attention_config'] = AttentionConfig(backend=desired_backend)
            backend_method = f'attention_config = AttentionConfig(backend={desired_backend})'
        except Exception as exc:
            print(f'  AttentionConfig import failed ({exc!r}); using env var fallback')
            os.environ['VLLM_ATTENTION_BACKEND'] = desired_backend
            backend_method = f'env VLLM_ATTENTION_BACKEND = {desired_backend}'
    else:
        os.environ['VLLM_ATTENTION_BACKEND'] = desired_backend
        backend_method = f'env VLLM_ATTENTION_BACKEND = {desired_backend}'
    os.environ.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '1')

spec_active = False
if cfg.enable_ngram_speculation:
    llm_kwargs['speculative_config'] = {
        'method': 'ngram',
        'num_speculative_tokens': cfg.speculative_num_tokens,
        'prompt_lookup_max': cfg.speculative_ngram_max,
        'prompt_lookup_min': cfg.speculative_ngram_min,
    }
    spec_active = True

print('\nvLLM init plan:')
print(f'  attention backend: {backend_method}')
print(f'  KV cache dtype:    {llm_kwargs.get("kv_cache_dtype", "default fp16")}')
print(f'  prefix caching:    {llm_kwargs.get("enable_prefix_caching", False)}')
print(f'  speculative dec.:  {"n-gram" if spec_active else "off"}')
print(f'  max_model_len:     {cfg.max_model_len}')
print(f'  max_num_seqs:      {cfg.max_num_seqs}')
print(f'  gpu_mem_util:      {cfg.gpu_memory_utilization}')

print('\nInitializing vLLM (this can take a minute)...')
try:
    llm = LLM(**llm_kwargs)
except Exception as exc:
    print(f'\nvLLM load FAILED: {type(exc).__name__}: {exc}')
    print('Diagnostics:')
    print('  - RESTART runtime and re-run from Cell 0 → Cell 6.')
    print(f'  - GPU SM {GPU_COMPUTE_CAPABILITY}; if FlashInfer was forced, set FLASHINFER_USABLE=False.')
    print('  - If FP8 KV crashes on init: set cfg.kv_cache_dtype = None and rerun Cell 2 → 6.')
    print('  - If spec decoding crashes: cfg.enable_ngram_speculation = False.')
    print('  - If OOM during graph capture: cfg.gpu_memory_utilization = 0.85, cfg.max_num_seqs = 256.')
    raise

chosen_backend = desired_backend or 'auto'
chosen_quantization = cfg.quantization
chosen_speculation = spec_active

print('\n' + '=' * 60)
print('vLLM loaded successfully.')
print(f'  attention backend (requested): {chosen_backend}')
print(f'  quantization:                  {chosen_quantization}')
print(f'  KV cache dtype:                {cfg.kv_cache_dtype}')
print(f'  speculative decoding:          {chosen_speculation}')
print('=' * 60)

try:
    import torch as _t
    globals()['torch'] = _t
    if _t.cuda.is_available():
        for i in range(_t.cuda.device_count()):
            free, total = _t.cuda.mem_get_info(i)
            print(f'After vLLM load GPU {i}: free={free/1024**3:.2f} GiB total={total/1024**3:.2f} GiB')
except Exception:
    pass

Loading tokenizer: Qwen/Qwen3-8B-AWQ
System+schema prefix size: ~1497 tokens (prefix-cached portion).
vLLM LLM signature (37 explicit params); attention_config=True

vLLM init plan:
  attention backend: auto-select (vLLM picks FA4 on Blackwell SM 12.x)
  KV cache dtype:    default fp16
  prefix caching:    True
  speculative dec.:  off
  max_model_len:     3072
  max_num_seqs:      384
  gpu_mem_util:      0.92

Initializing vLLM (this can take a minute)...
INFO 05-06 05:38:15 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/content/drive/MyDrive/swiss_law/models/huggingface', 'max_model_len': 3072, 'enable_prefix_caching': True, 'max_num_seqs': 384, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'disable_custom_all_reduce': True, 'model': 'Qwen/Qwen3-8B-AWQ'}
INFO 05-06 05:38:16 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-06 05:38:16 [model.py:1680] Using max model len 3072
INFO 05-06 05:38:16 [nixl_utils.py:20] Setting UCX_RC

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 05-06 05:38:17 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-06 05:38:17 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])

vLLM loaded successfully.
  attention backend (requested): auto
  quantization:                  awq_marlin
  KV cache dtype:                None
  speculative decoding:          False
After vLLM load GPU 0: free=6.90 GiB total=94.97 GiB


In [8]:
# Cell 7 — Generation helpers + warm-up

def render_prompt(row: dict, repair: bool = False, bad_output: str = '', error: str = '') -> str:
    user_prompt = build_user_prompt(row)
    if repair:
        user_prompt = f"""The previous output was invalid JSON.

Parser error:
{error}

Previous output:
{bad_output[:1400]}

Repair by returning exactly one complete compact JSON object using the same schema.
Do not add prose, anchors, citations, or retrieval views.

{user_prompt}"""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_raw(prompts: list[str], max_tokens: int) -> list[str]:
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    outputs = llm.generate(prompts, sampling_params=params, use_tqdm=False)
    return [out.outputs[0].text if out.outputs else '' for out in outputs]

def generate_raw_safe(prompts: list[str], max_tokens: int, min_split: int = 1) -> list[str]:
    try:
        return generate_raw(prompts, max_tokens)
    except Exception as exc:
        if len(prompts) <= min_split:
            raise
        mid = len(prompts) // 2
        print(f'  Batch of {len(prompts)} failed ({type(exc).__name__}); splitting into {mid}+{len(prompts)-mid}')
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        left = generate_raw_safe(prompts[:mid], max_tokens, min_split=min_split)
        right = generate_raw_safe(prompts[mid:], max_tokens, min_split=min_split)
        return left + right

def parse_or_retry(row: dict, raw: str | None) -> tuple[dict[str, Any], dict[str, Any]]:
    attempts = []
    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                raw = generate_raw_safe([render_prompt(row)], cfg.max_new_tokens)[0]
            obj = extract_json_object(raw)
            desc = normalize_descriptor(obj)
            return desc, {
                'status': 'ok' if attempt == 0 else 'ok_after_retry',
                'attempt_count': attempt + 1,
                'error': None,
                'raw_output': raw if cfg.include_raw_output_on_success else None,
            }
        except Exception as exc:
            err = repr(exc)
            attempts.append({'attempt': attempt + 1, 'error': err, 'raw_output': (raw or '')[:1400]})
            if attempt >= cfg.max_retries:
                return empty_descriptor(err), {
                    'status': 'failed_descriptor_parse',
                    'attempt_count': attempt + 1,
                    'error': err,
                    'attempts': attempts,
                    'raw_output': raw,
                }
            raw = generate_raw_safe(
                [render_prompt(row, repair=True, bad_output=raw or '', error=err)],
                cfg.retry_max_new_tokens,
            )[0]

print('Warm-up generation (16 short prompts)...')
_warm_row = {
    'citation': 'warmup',
    'law_title': 'warmup',
    'title_section_path': '',
    'structural': {'law_code': 'WARM', 'article': '1', 'units': []},
    'title_metadata': {'source_type': 'ordinance', 'enactment_year': '2020', 'enactment_date': ''},
    'static_hints': {'legal_area_static': '', 'domain_labels_en': []},
    'text': 'Warm-up paragraph for CUDA graph capture and FlashInfer kernel selection.',
}
_warm_prompts = [render_prompt(_warm_row)] * 16
_t = time.time()
_ = generate_raw_safe(_warm_prompts, max_tokens=64)
print(f'  done in {time.time()-_t:.2f}s')

Warm-up generation (16 short prompts)...
  done in 3.49s


In [9]:
cfg = Config()

if cfg.gpu_mode == 'single':
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == 'tp2':
    os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

base_dir = Path(cfg.base_dir)
data_dir = Path(cfg.data_dir)
out_dir = Path(cfg.output_dir)
local_scratch = Path(cfg.local_scratch_dir)
model_download_dir = Path(cfg.model_download_dir)
for d in [base_dir, data_dir, out_dir, model_download_dir, local_scratch]:
    d.mkdir(parents=True, exist_ok=True)

end_idx = cfg.start + cfg.limit - 1 if cfg.limit else -1
suffix = f'{cfg.start:07d}_{end_idx:07d}' if cfg.limit else f'{cfg.start:07d}_all'

output_jsonl          = out_dir / f'law_llm_descriptors_{suffix}.jsonl'
output_preview_csv    = out_dir / f'law_llm_descriptors_{suffix}_preview.csv'
output_failures_jsonl = out_dir / f'law_llm_descriptors_{suffix}_failures.jsonl'
output_metrics_json   = out_dir / f'law_llm_descriptors_{suffix}_metrics.json'

local_jsonl           = local_scratch / f'law_llm_descriptors_{suffix}.jsonl'
local_failures_jsonl  = local_scratch / f'law_llm_descriptors_{suffix}_failures.jsonl'

print(json.dumps(asdict(cfg), indent=2, default=str))
print('Output JSONL (final):', output_jsonl)
print('Output JSONL (hot)  :', local_jsonl)


{
  "base_dir": "/content/drive/MyDrive/swiss_law",
  "data_dir": "/content/drive/MyDrive/swiss_law/data",
  "output_dir": "/content/drive/MyDrive/swiss_law/outputs",
  "model_download_dir": "/content/drive/MyDrive/swiss_law/models/huggingface",
  "local_scratch_dir": "/content/drive/MyDrive/swiss_law/data/checkpoints",
  "input_jsonl": "/content/drive/MyDrive/swiss_law/data/law_llm_input.jsonl",
  "fallback_input_jsonl": "law_llm_input.jsonl",
  "model_name": "Qwen/Qwen3-8B-AWQ",
  "start": 0,
  "limit": 0,
  "sample_random": false,
  "random_seed": 42,
  "min_text_chars": 20,
  "max_text_chars": 1500,
  "gpu_mode": "single",
  "tensor_parallel_size": 1,
  "gpu_memory_utilization": 0.92,
  "max_model_len": 3072,
  "max_num_seqs": 384,
  "submit_chunk": 2048,
  "enforce_eager": false,
  "quantization": "awq_marlin",
  "disable_custom_all_reduce": true,
  "kv_cache_dtype": null,
  "enable_ngram_speculation": false,
  "speculative_num_tokens": 5,
  "speculative_ngram_min": 2,
  "speculat

In [ ]:
# Cell 8 — Run extraction (mega-batched, append-mode checkpointing)
#
# Submission strategy mirrors the court notebook:
#   submit cfg.submit_chunk (2048) prompts per llm.generate() call; vLLM
#   continuous-batches them across max_num_seqs (320) slots. Output appended
#   to local_jsonl after each chunk; copied to Drive at end.

records_for_preview = []
failures = []
status_counter = Counter()
role_counter = Counter()
t0 = time.time()
processed = 0
prompt_token_total = 0
output_token_total = 0
grounded_total = 0.0

# ── Checkpoint resume ────────────────────────────────────────────────────────
done_rows: set[int] = set()
if local_jsonl.exists():
    print(f'Checkpoint found: {local_jsonl}')
    with local_jsonl.open(encoding='utf-8') as _ckpt:
        for _line in _ckpt:
            _line = _line.strip()
            if not _line:
                continue
            try:
                done_rows.add(int(json.loads(_line)['_source_row']))
            except Exception:
                pass
    print(f'  → {len(done_rows):,} rows already done; will skip.')
else:
    print('No checkpoint found — starting fresh.')

remaining_df = work_df[~work_df['_source_row'].isin(done_rows)].reset_index(drop=True)
print(f'Rows remaining: {len(remaining_df):,} / {len(work_df):,} total ({len(done_rows):,} skipped).')
# ─────────────────────────────────────────────────────────────────────────────

def _generate_with_metrics(prompts: list[str], max_tokens: int):
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    return llm.generate(prompts, sampling_params=params, use_tqdm=False)

def _generate_with_metrics_safe(prompts: list[str], max_tokens: int, min_split: int = 1):
    try:
        return _generate_with_metrics(prompts, max_tokens)
    except Exception as exc:
        if len(prompts) <= min_split:
            raise
        mid = len(prompts) // 2
        print(f'  Batch of {len(prompts)} failed ({type(exc).__name__}); splitting into {mid}+{len(prompts)-mid}')
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        left = _generate_with_metrics_safe(prompts[:mid], max_tokens, min_split=min_split)
        right = _generate_with_metrics_safe(prompts[mid:], max_tokens, min_split=min_split)
        return left + right

N = len(remaining_df)
SUBMIT = cfg.submit_chunk
print(f'Processing {N:,} remaining rows in chunks of {SUBMIT} (max_num_seqs={cfg.max_num_seqs}; {len(done_rows):,} done).')

with local_jsonl.open('a', encoding='utf-8') as out_f:
    pbar = tqdm(total=N, desc='Law LLM enrichment', smoothing=0.05)

    for start in range(0, N, SUBMIT):
        end = min(start + SUBMIT, N)
        chunk = remaining_df.iloc[start:end]

        row_objs = []
        prompts = []
        for _, row in chunk.iterrows():
            row_obj = {
                '_source_row': int(row['_source_row']),
                'citation': row['citation'],
                'law_title': row['law_title'],
                'title_section_path': row['title_section_path'],
                'structural': row['structural'],
                'title_metadata': row['title_metadata'],
                'static_hints': row['static_hints'],
                'llm_priority': row['llm_priority'],
                'text': row['text'],
            }
            row_objs.append(row_obj)
            prompts.append(render_prompt(row_obj))

        chunk_t = time.time()
        try:
            req_outputs = _generate_with_metrics_safe(prompts, cfg.max_new_tokens)
        except Exception as exc:
            print(f'Chunk {start}-{end} failed completely; falling back to per-row: {exc!r}')
            req_outputs = [None] * len(row_objs)

        for row_obj, req_out in zip(row_objs, req_outputs):
            if req_out is None:
                raw = None
            else:
                raw = req_out.outputs[0].text if req_out.outputs else ''
                try:
                    prompt_token_total += len(req_out.prompt_token_ids or [])
                    if req_out.outputs:
                        output_token_total += len(req_out.outputs[0].token_ids or [])
                except Exception:
                    pass

            desc, gen = parse_or_retry(row_obj, raw)
            grounded = grounded_terms_pct(desc.get('terms_de_to_en', []), row_obj['text'])
            grounded_total += grounded
            role = desc.get('provision_role_llm', 'other')
            role_counter[role] += 1

            rec = {
                '_source_row': row_obj['_source_row'],
                'citation': row_obj['citation'],
                'language': 'de',
                'llm_priority': row_obj['llm_priority'],
                'llm_enrichment': desc,
                'llm_quality': {
                    'json_valid': gen['status'].startswith('ok'),
                    'terms_grounded_pct': grounded,
                    'boilerplate_role': role in BOILERPLATE_ROLES,
                },
                'llm_generation': {
                    'model': cfg.model_name,
                    'method': 'law_descriptor_v1',
                    'attention_backend': chosen_backend,
                    'kv_cache_dtype': cfg.kv_cache_dtype,
                    'speculative': chosen_speculation,
                    **gen,
                },
            }
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            processed += 1
            status_counter[gen['status']] += 1
            if len(records_for_preview) < 1000:
                records_for_preview.append(rec)
            if gen['status'].startswith('failed'):
                failures.append(rec)

        out_f.flush()
        pbar.update(end - start)

        rate = end / max(time.time() - t0, 1e-9)
        chunk_rate = (end - start) / max(time.time() - chunk_t, 1e-9)
        pbar.set_postfix(
            chunk_rps=f'{chunk_rate:.1f}',
            avg_rps=f'{rate:.1f}',
            ok=status_counter['ok'],
            failed=status_counter['failed_descriptor_parse'],
        )

    pbar.close()

elapsed = time.time() - t0
print(f'\nFinished {processed} rows in {elapsed:.1f}s = {processed/max(elapsed,1e-9):.2f} rows/sec')
print(f'Prompt tokens total:  {prompt_token_total:,}  (avg {prompt_token_total/max(processed,1):.0f}/row)')
print(f'Output tokens total:  {output_token_total:,}  (avg {output_token_total/max(processed,1):.0f}/row)')
print(f'Avg terms_grounded_pct: {grounded_total/max(processed,1):.3f}')

print(f'\nCopying {local_jsonl} → {output_jsonl} ...')
shutil.copy2(local_jsonl, output_jsonl)
print('  done.')

if failures:
    with local_failures_jsonl.open('w', encoding='utf-8') as f:
        for rec in failures:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    shutil.copy2(local_failures_jsonl, output_failures_jsonl)
elif output_failures_jsonl.exists():
    output_failures_jsonl.unlink()

preview_rows = []
for rec in records_for_preview:
    e = rec['llm_enrichment']
    g = rec['llm_generation']
    preview_rows.append({
        '_source_row': rec['_source_row'],
        'citation': rec['citation'],
        'status': g['status'],
        'role': e.get('provision_role_llm'),
        'specificity_score': e.get('specificity_score'),
        'english_summary': e.get('english_summary'),
        'legal_rule': e.get('legal_rule'),
        'legal_question': e.get('legal_question'),
        'concepts_en': ' | '.join(e.get('concepts_en', [])),
        'terms_de_to_en': ' | '.join(f"{t['de']}→{t['en']}" for t in e.get('terms_de_to_en', [])),
        'addressees': ' | '.join(e.get('addressees', [])),
        'sanctions_or_consequences': ' | '.join(e.get('sanctions_or_consequences', [])),
        'applicability_conditions': ' | '.join(e.get('applicability_conditions', [])),
        'exceptions_or_limitations': ' | '.join(e.get('exceptions_or_limitations', [])),
        'terms_grounded_pct': rec['llm_quality']['terms_grounded_pct'],
    })
preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(output_preview_csv, index=False)

metrics = {
    'start': cfg.start,
    'limit': cfg.limit,
    'selected_rows': len(work_df),
    'written_rows_this_session': processed,
    'total_rows_in_file': len(done_rows) + processed,
    'failures': len(failures),
    'elapsed_seconds': elapsed,
    'rows_per_second': processed / max(elapsed, 1e-9),
    'prompt_tokens_total': prompt_token_total,
    'output_tokens_total': output_token_total,
    'tokens_per_second': (prompt_token_total + output_token_total) / max(elapsed, 1e-9),
    'output_tokens_per_second': output_token_total / max(elapsed, 1e-9),
    'avg_terms_grounded_pct': grounded_total / max(processed, 1),
    'attention_backend': chosen_backend,
    'quantization': chosen_quantization,
    'kv_cache_dtype': cfg.kv_cache_dtype,
    'speculative': chosen_speculation,
    'status_counts': dict(status_counter),
    'provision_role_distribution': dict(role_counter),
    'output_jsonl': str(output_jsonl),
    'output_preview_csv': str(output_preview_csv),
    'output_failures_jsonl': str(output_failures_jsonl) if failures else None,
    'config': {k: (str(v) if not isinstance(v, (int, float, bool, str, list, dict, type(None))) else v)
               for k, v in asdict(cfg).items()},
}
output_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(metrics, indent=2, default=str))
display(preview_df.head(50))

No checkpoint found — starting fresh.
Rows remaining: 173,033 / 173,033 total (0 skipped).
Processing 173,033 remaining rows in chunks of 2048 (max_num_seqs=384; 0 done).


Law LLM enrichment:   0%|          | 0/173033 [00:00<?, ?it/s]

In [ ]:
# Cell 9 — QC: verify schema integrity, term grounding, and absence of forbidden fields.

FORBIDDEN = {
    'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
    'outcome_signal', 'enrichment_quality', 'anchor_quality_flags',
    'incoming_references', 'outgoing_references', 'adjacent_citations',
    'law_code', 'law_title', 'enactment_date', 'enactment_year',
}

def find_forbidden(obj: Any, path: str = '') -> list[str]:
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{path}.{k}' if path else k
            if k in FORBIDDEN and path == 'llm_enrichment':
                hits.append(p)
            hits.extend(find_forbidden(v, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_forbidden(v, f'{path}[{i}]'))
    return hits

qc = []
with output_jsonl.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue
        rec = json.loads(line)
        e = rec['llm_enrichment']
        qc.append({
            'citation': rec['citation'],
            'status': rec['llm_generation']['status'],
            'role': e.get('provision_role_llm'),
            'forbidden_fields': find_forbidden({'llm_enrichment': e}),
            'has_summary': bool(e.get('english_summary')),
            'has_rule': bool(e.get('legal_rule')),
            'has_question': bool(e.get('legal_question')),
            'concept_count': len(e.get('concepts_en', [])),
            'term_pair_count': len(e.get('terms_de_to_en', [])),
            'defined_term_count': len(e.get('defined_terms', [])),
            'addressee_count': len(e.get('addressees', [])),
            'specificity_score': e.get('specificity_score'),
            'terms_grounded_pct': rec['llm_quality']['terms_grounded_pct'],
        })
qc_df = pd.DataFrame(qc)
display(qc_df.head(100))
print('Rows checked:', len(qc_df))
print('Forbidden field rows:', int(qc_df['forbidden_fields'].apply(bool).sum()))
print('Failed rows:', int(qc_df['status'].str.startswith('failed').sum()))
print('Status counts:', qc_df['status'].value_counts().to_dict())
print('Provision role distribution:', qc_df['role'].value_counts().to_dict())
print(f'Avg terms_grounded_pct: {qc_df["terms_grounded_pct"].mean():.3f}')
print(f'Median concept_count: {qc_df["concept_count"].median():.1f}')
print(f'Median term_pair_count: {qc_df["term_pair_count"].median():.1f}')
print(f'Boilerplate-role rows: {int((qc_df["role"].isin(list(BOILERPLATE_ROLES))).sum())}')
print(f'Substantive rows with non-empty rule: {int((~qc_df["role"].isin(list(BOILERPLATE_ROLES)) & qc_df["has_rule"]).sum())}')

## Tuning notes

- **`max_text_chars`**: defaulted to 1500. Law p99 = 1274 chars and median = 181, so >99% of rows fit untruncated. Bump to 2000 if you start seeing too many TRUNCATED markers in the preview CSV.
- **`max_new_tokens`**: 700 covers ~12 fields with healthy lists. If you see truncated JSON output (parse failures with no closing brace), bump to 800 and `retry_max_new_tokens` to 950.
- **`temperature`** = 0.1 (vs the court run's 0.0). Reason: the law schema requires creative legal-English mapping (`Bewilligung`→"permit / authorisation"), where strict greedy decoding tends to latch onto literal translations. 0.1 is small enough to keep JSON deterministic in practice but breaks the literal-translation attractor.
- **Boilerplate suppression** happens twice: in `normalize_descriptor()` (so the descriptor file is clean) and again in `scripts/merge_law_llm_into_v1_cards.py` (so even pre-existing dirty rows get scrubbed at merge time).
- **Resume**: kill the cell and re-run; rows already in `local_jsonl` (and `output_jsonl` once copied) are skipped via `_source_row`.
- **Next step after this notebook completes**: download `output_jsonl` and run
  `python scripts/merge_law_llm_into_v1_cards.py --llm <path>` to produce `artifacts/law_authority_cards_v2_unified.jsonl`.